In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd /content/drive/MyDrive/SynDataGen/models/RS-paint

/content/drive/.shortcut-targets-by-id/1qW2CVvnw_L-G4Gb2oBNdqjYnP62j_EzG/SynDataGen-Drive/models/RS-paint


In [3]:
!ls -la

total 1
-rw------- 1 root root 529 Jul 14 10:51 Untitled0.ipynb


In [4]:
!git clone https://github.com/SteveImmanuel/rs-paint.git .

fatal: destination path '.' already exists and is not an empty directory.


In [5]:
!git clone https://github.com/SteveImmanuel/rs-paint.git tmp_repo
!mv tmp_repo/* tmp_repo/.[!.]* . 2>/dev/null
!rmdir tmp_repo

Cloning into 'tmp_repo'...
remote: Enumerating objects: 317, done.
remote: Counting objects: 100% (317/317), done.
remote: Compressing objects: 100% (238/238), done.
remote: Total 317 (delta 124), reused 254 (delta 74), pack-reused 0 (from 0)
Receiving objects: 100% (317/317), 11.82 MiB | 13.07 MiB/s, done.
Resolving deltas: 100% (124/124), done.


In [6]:
!ls -la

total 92
drwx------ 2 root root  4096 Jul 14 10:53 configs
-rw------- 1 root root  7374 Jul 14 10:53 environment.yaml
drwx------ 5 root root  4096 Jul 14 10:53 eval_tool
drwx------ 3 root root  4096 Jul 14 10:53 figure
drwx------ 8 root root  4096 Jul 14 10:53 .git
-rw------- 1 root root  3495 Jul 14 10:53 .gitignore
-rw------- 1 root root   148 Jul 14 10:53 inference_test_bench.sh
drwx------ 5 root root  4096 Jul 14 10:53 ldm
-rw------- 1 root root 11356 Jul 14 10:53 LICENSE
-rw------- 1 root root 27105 Jul 14 10:53 main.py
drwx------ 2 root root  4096 Jul 14 10:53 notebooks
-rw------- 1 root root  5147 Jul 14 10:53 README.md
drwx------ 2 root root  4096 Jul 14 10:53 scripts
-rw------- 1 root root   233 Jul 14 10:53 setup.py
drwx------ 2 root root  4096 Jul 14 10:53 test_bench
-rw------- 1 root root   832 Jul 14 10:53 test.sh
-rw------- 1 root root   166 Jul 14 10:53 train.sh
-rw------- 1 root root  2444 Jul 14 10:52 Untitled0.ipynb


In [7]:
!cat README.md

# Tackling Few-Shot Segmentation in Remote Sensing via Inpainting Diffusion Model

**ICLR Machine Learning for Remote Sensing Workshop, 2025 (Best Paper Award)**


This repo contains the official code for training and generation for the paper "Tackling Few-Shot Segmentation in Remote Sensing via Inpainting Diffusion Model".


<!-- [![Hugging Face Demo](https://img.shields.io/badge/🤗%20Hugging%20Face%20(LCM)-Space-yellow)](https://huggingface.co/spaces/prs-eth/marigold-lcm) -->
[![Website](figure/badge-website.svg)](https://steveimmanuel.github.io/rs-paint)
[![Paper](https://img.shields.io/badge/arXiv-PDF-b31b1b)](https://arxiv.org/abs/2503.03785)
[![License](https://img.shields.io/badge/License-Apache--2.0-929292)](https://www.apache.org/licenses/LICENSE-2.0)

[Steve Andreas Immanuel](https://steveimm.id) | [Woojin Cho](https://woojin-cho.github.io/) | [Junhyuk Heo](https://rokmc1250.github.io/) | [Darongsae Kwon](https://www.linkedin.com/in/darongsaekwon).

We introduce a image-condit

In [8]:
!mkdir -p checkpoints

In [9]:
!wget -O checkpoints/sd_inpaint_samrs_ep74.ckpt "https://huggingface.co/SteveImmanuel/RSPaint/resolve/main/sd_inpaint_samrs_ep74.ckpt"
!wget -O checkpoints/remoteclip.pt "https://huggingface.co/SteveImmanuel/RSPaint/resolve/main/remoteclip.pt"

--2026-07-14 10:56:48--  https://huggingface.co/SteveImmanuel/RSPaint/resolve/main/sd_inpaint_samrs_ep74.ckpt
Resolving huggingface.co (huggingface.co)... 3.170.185.25, 3.170.185.35, 3.170.185.14, ...
Connecting to huggingface.co (huggingface.co)|3.170.185.25|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://cas-bridge.xethub.hf.co/xet-bridge-us/67c7f72f02935d02b0bf907f/46412f59dcf3207bf53b3f00d9a1a1d64624764e55e203d0a7752775e05feaa5?Expires=1784030208&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly9jYXMtYnJpZGdlLnhldGh1Yi5oZi5jby94ZXQtYnJpZGdlLXVzLzY3YzdmNzJmMDI5MzVkMDJiMGJmOTA3Zi80NjQxMmY1OWRjZjMyMDdiZjUzYjNmMDBkOWExYTFkNjQ2MjQ3NjRlNTVlMjAzZDBhNzc1Mjc3NWUwNWZlYWE1KiIsIkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTc4NDAzMDIwOH19fV19&Signature=MEQCIBuoojSFB4%7ERqbIr%7EPCuKIWeYVywopp9g3bqOUdkvd31AiBf5X1tgs2dPNHMtFb8Y1jBW3ZWBjuDneNcbqgcUwxOSA__&Key-Pair-Id=K3EPXBYC3CKDRZ&response-content-disposition=inline%3B+filename*%3DUTF-8%27%

In [10]:
!pip install -q huggingface_hub

In [13]:
!rm -f checkpoints/remoteclip.pt

In [14]:
!curl -L --retry 5 --retry-delay 3 -A "Mozilla/5.0" \
    -o checkpoints/remoteclip.pt \
    "https://huggingface.co/SteveImmanuel/RSPaint/resolve/main/remoteclip.pt"

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   848  100   848    0     0   5053      0 --:--:-- --:--:-- --:--:--  5077
100    48  100    48    0     0    167      0 --:--:-- --:--:-- --:--:--  2400


In [15]:
!cat checkpoints/remoteclip.pt

Auth failed: SignatureError: invalid key pair id

In [16]:
!curl -L --retry 5 --retry-delay 5 -A "Mozilla/5.0" \
    -o checkpoints/remoteclip.pt \
    "https://huggingface.co/SteveImmanuel/RSPaint/resolve/main/remoteclip.pt?download=true"

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1010  100  1010    0     0   5678      0 --:--:-- --:--:-- --:--:--  5706
100 1631M  100 1631M    0     0  42.0M      0  0:00:38  0:00:38 --:--:-- 53.4M


In [17]:
!ls -lh checkpoints/

total 6.5G
-rw------- 1 root root 1.6G Jul 14 11:31 remoteclip.pt
-rw------- 1 root root 4.9G Jul 14 10:59 sd_inpaint_samrs_ep74.ckpt


In [18]:
!pip install -q omegaconf pytorch-lightning einops open_clip_torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.6 MB/s eta 0:00:00


In [21]:
%%writefile run_rspaint_single.py
"""
run_rspaint_single.py
======================
run_harmonidiff_single.py'nin RS-Paint'e UYARLANMIS hali. Ayni instance
klasor yapisini (bg.png / mask_location.png / fg.png / meta.json) okur,
sadece MODEL COGRISI HarmoniDiff yerine RS-Paint (Paint-by-Example tabanli
Stable Diffusion inpainting + RemoteCLIP) olur.
"""
import argparse
import json
import os
import shutil

import numpy as np
import torch
import torch.nn.functional as F
import torchvision
from omegaconf import OmegaConf
from PIL import Image
from pytorch_lightning import seed_everything
from torchvision.transforms import Resize

from ldm.util import instantiate_from_config
from ldm.models.diffusion.plms import PLMSSampler


def load_instance(inst_dir):
    bg = Image.open(os.path.join(inst_dir, "bg.png")).convert("RGB")
    mask = Image.open(os.path.join(inst_dir, "mask_location.png")).convert("L")
    fg = Image.open(os.path.join(inst_dir, "fg.png")).convert("RGB")
    with open(os.path.join(inst_dir, "meta.json")) as f:
        meta = json.load(f)
    return bg, mask, fg, meta


def load_model_from_config(config, ckpt, device):
    print(f"Model yukleniyor: {ckpt}")
    pl_sd = torch.load(ckpt, map_location="cpu")
    sd = pl_sd["state_dict"]
    model = instantiate_from_config(config.model)
    model.load_state_dict(sd, strict=False)
    model = model.to(device)
    model.eval()
    return model


def get_tensor():
    return torchvision.transforms.Compose([
        torchvision.transforms.ToTensor(),
        torchvision.transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
    ])


def get_tensor_clip():
    return torchvision.transforms.Compose([
        torchvision.transforms.ToTensor(),
        torchvision.transforms.Normalize((0.48145466, 0.4578275, 0.40821073),
                                          (0.26862954, 0.26130258, 0.27577711)),
    ])


@torch.inference_mode()
def rspaint_generate(model, sampler, bg_img, mask_img, ref_img, scale, ddim_steps, n_samples, device):
    img_p = bg_img.resize((512, 512))
    image_tensor = get_tensor()(img_p).unsqueeze(0).repeat(n_samples, 1, 1, 1).to(device)

    ref_p = ref_img.resize((224, 224))
    ref_tensor = get_tensor_clip()(ref_p).unsqueeze(0).repeat(n_samples, 1, 1, 1).to(device)

    mask = np.array(mask_img.resize((512, 512), Image.NEAREST))[None, None]
    mask = 1 - mask.astype(np.float32) / 255.0
    mask[mask < 0.5] = 0
    mask[mask >= 0.5] = 1
    mask_tensor = torch.from_numpy(mask).repeat(n_samples, 1, 1, 1).to(device)

    uc = model.learnable_vector.repeat(n_samples, 1, 1) if scale != 1.0 else None
    c = model.proj_out(model.get_learned_conditioning(ref_tensor))

    inpaint_image = image_tensor * mask_tensor
    z_inpaint = model.get_first_stage_encoding(model.encode_first_stage(inpaint_image)).detach()
    test_model_kwargs = {
        "inpaint_image": z_inpaint,
        "inpaint_mask": Resize([z_inpaint.shape[-2], z_inpaint.shape[-1]])(mask_tensor),
    }

    samples_ddim, _ = sampler.sample(
        S=ddim_steps, conditioning=c, batch_size=n_samples, shape=[4, 64, 64],
        verbose=False, unconditional_guidance_scale=scale, unconditional_conditioning=uc,
        eta=0.0, x_T=None, test_model_kwargs=test_model_kwargs,
    )
    x = model.decode_first_stage(samples_ddim)
    x = torch.clamp((x + 1.0) / 2.0, min=0.0, max=1.0)
    return x.cpu().permute(0, 2, 3, 1).numpy()


def pick_best_sample(model, samples, ref_img, device):
    img_preprocessor = model.cond_stage_model.preprocess
    ref_feat = model.cond_stage_model.get_visual_clip_features(
        img_preprocessor(ref_img).unsqueeze(0).to(device)
    )
    best_idx, best_score = 0, -1.0
    for i in range(len(samples)):
        result = Image.fromarray((samples[i] * 255).astype(np.uint8))
        result_feat = model.cond_stage_model.get_visual_clip_features(
            img_preprocessor(result).unsqueeze(0).to(device)
        )
        score = F.cosine_similarity(ref_feat, result_feat, dim=-1).item()
        if score > best_score:
            best_score, best_idx = score, i
    return best_idx, best_score


def harmonize_one_rspaint(model, sampler, inst_dir, out_dir, scale, ddim_steps, n_samples, device):
    bg, mask, fg,

Overwriting run_rspaint_single.py


In [22]:
!wc -l run_rspaint_single.py

114 run_rspaint_single.py


In [23]:
!python run_rspaint_single.py \
    --synthetic_dir /content/drive/MyDrive/SynDataGen/outputs/synthetic_pairs \
    --out_dir /content/drive/MyDrive/SynDataGen/outputs/rspaint_results \
    --instances haiti-earthquake_00000000_inst0001

In [24]:
!ls -la /content/drive/MyDrive/SynDataGen/outputs/rspaint_results/

ls: cannot access '/content/drive/MyDrive/SynDataGen/outputs/rspaint_results/': No such file or directory


In [25]:
import subprocess
result = subprocess.run(
    ["python", "run_rspaint_single.py",
     "--synthetic_dir", "/content/drive/MyDrive/SynDataGen/outputs/synthetic_pairs",
     "--out_dir", "/content/drive/MyDrive/SynDataGen/outputs/rspaint_results",
     "--instances", "haiti-earthquake_00000000_inst0001"],
    capture_output=True, text=True
)
print("STDOUT:\n", result.stdout)
print("STDERR:\n", result.stderr)
print("RETURN CODE:", result.returncode)

STDOUT:
 
STDERR:
 
RETURN CODE: 0


In [26]:
!tail -20 run_rspaint_single.py


def pick_best_sample(model, samples, ref_img, device):
    img_preprocessor = model.cond_stage_model.preprocess
    ref_feat = model.cond_stage_model.get_visual_clip_features(
        img_preprocessor(ref_img).unsqueeze(0).to(device)
    )
    best_idx, best_score = 0, -1.0
    for i in range(len(samples)):
        result = Image.fromarray((samples[i] * 255).astype(np.uint8))
        result_feat = model.cond_stage_model.get_visual_clip_features(
            img_preprocessor(result).unsqueeze(0).to(device)
        )
        score = F.cosine_similarity(ref_feat, result_feat, dim=-1).item()
        if score > best_score:
            best_score, best_idx = score, i
    return best_idx, best_score


def harmonize_one_rspaint(model, sampler, inst_dir, out_dir, scale, ddim_steps, n_samples, device):
    bg, mask, fg,


In [27]:
!rm run_rspaint_single.py

In [28]:
%%writefile run_rspaint_single.py
import argparse
import json
import os
import shutil

import numpy as np
import torch
import torch.nn.functional as F
import torchvision
from omegaconf import OmegaConf
from PIL import Image
from pytorch_lightning import seed_everything
from torchvision.transforms import Resize

from ldm.util import instantiate_from_config
from ldm.models.diffusion.plms import PLMSSampler


def load_instance(inst_dir):
    bg = Image.open(os.path.join(inst_dir, "bg.png")).convert("RGB")
    mask = Image.open(os.path.join(inst_dir, "mask_location.png")).convert("L")
    fg = Image.open(os.path.join(inst_dir, "fg.png")).convert("RGB")
    with open(os.path.join(inst_dir, "meta.json")) as f:
        meta = json.load(f)
    return bg, mask, fg, meta


def load_model_from_config(config, ckpt, device):
    print(f"Model yukleniyor: {ckpt}")
    pl_sd = torch.load(ckpt, map_location="cpu")
    sd = pl_sd["state_dict"]
    model = instantiate_from_config(config.model)
    model.load_state_dict(sd, strict=False)
    model = model.to(device)
    model.eval()
    return model


def get_tensor():
    return torchvision.transforms.Compose([
        torchvision.transforms.ToTensor(),
        torchvision.transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
    ])


def get_tensor_clip():
    return torchvision.transforms.Compose([
        torchvision.transforms.ToTensor(),
        torchvision.transforms.Normalize((0.48145466, 0.4578275, 0.40821073),
                                          (0.26862954, 0.26130258, 0.27577711)),
    ])


@torch.inference_mode()
def rspaint_generate(model, sampler, bg_img, mask_img, ref_img, scale, ddim_steps, n_samples, device):
    img_p = bg_img.resize((512, 512))
    image_tensor = get_tensor()(img_p).unsqueeze(0).repeat(n_samples, 1, 1, 1).to(device)

    ref_p = ref_img.resize((224, 224))
    ref_tensor = get_tensor_clip()(ref_p).unsqueeze(0).repeat(n_samples, 1, 1, 1).to(device)

    mask = np.array(mask_img.resize((512, 512), Image.NEAREST))[None, None]
    mask = 1 - mask.astype(np.float32) / 255.0
    mask[mask < 0.5] = 0
    mask[mask >= 0.5] = 1
    mask_tensor = torch.from_numpy(mask).repeat(n_samples, 1, 1, 1).to(device)

    uc = model.learnable_vector.repeat(n_samples, 1, 1) if scale != 1.0 else None
    c = model.proj_out(model.get_learned_conditioning(ref_tensor))

    inpaint_image = image_tensor * mask_tensor
    z_inpaint = model.get_first_stage_encoding(model.encode_first_stage(inpaint_image)).detach()
    test_model_kwargs = {
        "inpaint_image": z_inpaint,
        "inpaint_mask": Resize([z_inpaint.shape[-2], z_inpaint.shape[-1]])(mask_tensor),
    }

    samples_ddim, _ = sampler.sample(
        S=ddim_steps, conditioning=c, batch_size=n_samples, shape=[4, 64, 64],
        verbose=False, unconditional_guidance_scale=scale, unconditional_conditioning=uc,
        eta=0.0, x_T=None, test_model_kwargs=test_model_kwargs,
    )
    x = model.decode_first_stage(samples_ddim)
    x = torch.clamp((x + 1.0) / 2.0, min=0.0, max=1.0)
    return x.cpu().permute(0, 2, 3, 1).numpy()


def pick_best_sample(model, samples, ref_img, device):
    img_preprocessor = model.cond_stage_model.preprocess
    ref_feat = model.cond_stage_model.get_visual_clip_features(
        img_preprocessor(ref_img).unsqueeze(0).to(device)
    )
    best_idx, best_score = 0, -1.0
    for i in range(len(samples)):
        result = Image.fromarray((samples[i] * 255).astype(np.uint8))
        result_feat = model.cond_stage_model.get_visual_clip_features(
            img_preprocessor(result).unsqueeze(0).to(device)
        )
        score = F.cosine_similarity(ref_feat, result_feat, dim=-1).item()
        if score > best_score:
            best_score, best_idx = score, i
    return best_idx, best_score


def harmonize_one_rspaint(model, sampler, inst_dir, out_dir, scale, ddim_steps, n_samples, device):
    bg, mask, fg, meta = load_instance(inst_dir)

    samples = rspaint_generate(model, sampler, bg, mask, fg, scale, ddim_steps, n_samples, device)
    best_idx, best_score = pick_best_sample(model, samples, fg, device)
    result_img = Image.fromarray((samples[best_idx] * 255).astype(np.uint8)).resize(bg.size)

    inst_name = os.path.basename(inst_dir.rstrip("/"))
    inst_out_dir = os.path.join(out_dir, inst_name)
    os.makedirs(inst_out_dir, exist_ok=True)
    result_img.save(os.path.join(inst_out_dir, "result_raw.png"))
    shutil.copy(os.path.join(inst_dir, "meta.json"), os.path.join(inst_out_dir, "meta.json"))
    with open(os.path.join(inst_out_dir, "clip_similarity.txt"), "w") as f:
        f.write(f"best_sample_idx={best_idx}\nclip_cosine_similarity={best_score:.5f}\n")

    print(f"OK: {inst_name} -> {inst_out_dir}/result_raw.png (clip_sim={best_score:.4f})")
    return inst_out_dir


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--synthetic_dir", type=str, required=True)
    parser.add_argument("--out_dir", type=str, required=True)
    parser.add_argument("--instances", type=str, nargs="+", required=True)
    parser.add_argument("--config_path", type=str, default="configs/rs_remoteclip.yaml")
    parser.add_argument("--ckpt_path", type=str, default="checkpoints/sd_inpaint_samrs_ep74.ckpt")
    parser.add_argument("--scale", type=float, default=8.0)
    parser.add_argument("--ddim_steps", type=int, default=50)
    parser.add_argument("--n_samples", type=int, default=4)
    parser.add_argument("--seed", type=int, default=20250110)
    args = parser.parse_args()

    os.makedirs(args.out_dir, exist_ok=True)
    seed_everything(args.seed)
    device = 0 if torch.cuda.is_available() else "cpu"

    config = OmegaConf.load(args.config_path)
    model = load_model_from_config(config, args.ckpt_path, device)
    sampler = PLMSSampler(model)
    print("Model hazir.")

    for inst_name in args.instances:
        inst_dir = os.path.join(args.synthetic_dir, inst_name)
        if not os.path.isdir(inst_dir):
            print(f"UYARI: klasor bulunamadi, atlaniyor: {inst_dir}")
            continue
        try:
            harmonize_one_rspaint(model, sampler, inst_dir, args.out_dir,
                                   args.scale, args.ddim_steps, args.n_samples, device)
        except Exception as e:
            print(f"HATA - {inst_name}: {e}")


if __name__ == "__main__":
    main()

Writing run_rspaint_single.py


In [29]:
!wc -l run_rspaint_single.py
!tail -5 run_rspaint_single.py

159 run_rspaint_single.py
            print(f"HATA - {inst_name}: {e}")


if __name__ == "__main__":
    main()


In [30]:
import subprocess
result = subprocess.run(
    ["python", "run_rspaint_single.py",
     "--synthetic_dir", "/content/drive/MyDrive/SynDataGen/outputs/synthetic_pairs",
     "--out_dir", "/content/drive/MyDrive/SynDataGen/outputs/rspaint_results",
     "--instances", "haiti-earthquake_00000000_inst0001"],
    capture_output=True, text=True
)
print("STDOUT:\n", result.stdout)
print("STDERR:\n", result.stderr)
print("RETURN CODE:", result.returncode)

STDOUT:
 Model yukleniyor: checkpoints/sd_inpaint_samrs_ep74.ckpt

STDERR:
 Seed set to 20250110
Traceback (most recent call last):
  File "/content/drive/.shortcut-targets-by-id/1qW2CVvnw_L-G4Gb2oBNdqjYnP62j_EzG/private-bdi/models/RS-paint/run_rspaint_single.py", line 159, in <module>
    main()
  File "/content/drive/.shortcut-targets-by-id/1qW2CVvnw_L-G4Gb2oBNdqjYnP62j_EzG/private-bdi/models/RS-paint/run_rspaint_single.py", line 142, in main
    model = load_model_from_config(config, args.ckpt_path, device)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/drive/.shortcut-targets-by-id/1qW2CVvnw_L-G4Gb2oBNdqjYnP62j_EzG/private-bdi/models/RS-paint/run_rspaint_single.py", line 32, in load_model_from_config
    model = instantiate_from_config(config.model)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/drive/.shortcut-targets-by-id/1qW2CVvnw_L-G4Gb2oBNdqjYnP62j_EzG/private-bdi/models/RS-paint/ldm/util.py", line 85, in instantiate_fr

In [31]:
!grep -i "pytorch-lightning\|pytorch_lightning" environment.yaml

      - pytorch-lightning==1.4.2
